# Long-context multi-task transformer training

This notebook fine-tunes an open-source transformer on `train.xlsx` only. It jointly learns:

1. **Task 1A:** four-class suicide-risk prediction.
2. **Task 1B:** token-level evidence extraction copied from the source post.
3. **Task 2:** 24-label suicide-factor prediction.

The validation set is separated by `anon_user_id`. `leaderboard.xlsx` is not loaded or predicted here.

## Why this design

- `answerdotai/ModernBERT-base` is an Apache-2.0 encoder trained for long context. The default 1,024-token window retains much more of long Reddit posts than a 512-token encoder.
- One shared encoder transfers information between risk, evidence, and factors.
- Evidence is a learned token-tagging task, not a keyword rule.
- Factor training uses asymmetric loss to reduce domination by easy negative labels.
- Per-factor thresholds and the evidence threshold are tuned on validation predictions.
- The checkpoint is selected by the exact local composite: `0.4*risk + 0.3*evidence + 0.3*factors`.

In [1]:
# Run once if this environment is missing dependencies.
# %pip install -U pandas openpyxl numpy scikit-learn torch transformers accelerate


In [2]:
from pathlib import Path
import json
import pandas as pd
import torch

from transformer_multitask import (
    Config, FACTOR_LABELS, RISK_LABELS, choose_device,
    load_training_frame, grouped_split, train_one_fold,
)

TRAIN_PATH = Path('train.xlsx')
assert TRAIN_PATH.exists(), f'Missing {TRAIN_PATH.resolve()}'
print('PyTorch:', torch.__version__)
print('Device:', choose_device())

PyTorch: 2.13.0
Device: mps


## Validate labels and evidence annotations

This removes the annotation prefix `Main phrases that lead to this assessment:` and treats `none` as empty evidence. It then locates each remaining phrase inside the original post to create token supervision.

In [3]:
frame = load_training_frame(TRAIN_PATH)
print('Rows:', len(frame))
print('Authors:', frame['anon_user_id'].nunique())
print('Gold evidence phrases:', frame['evidence_spans'].map(len).sum())
print('Evidence phrases not located in post:', frame['unlocated_evidence'].map(len).sum())
display(frame['risk_level'].value_counts().reindex(RISK_LABELS).to_frame('count'))
display(frame.loc[frame['unlocated_evidence'].map(bool),
                  ['row_id', 'evidence_spans', 'unlocated_evidence']].head(20))

Rows: 1635
Authors: 153
Gold evidence phrases: 1827
Evidence phrases not located in post: 36


,count
risk_level,
Indicator,611
Ideation,519
Behavior,391
Attempt,114


,row_id,evidence_spans,unlocated_evidence
248,P00371,[last night's suicide attempt],[last night's suicide attempt]
380,P00576,"[dont want... the past suicide attempts, be ok...",[the cloudds will pass]
591,P00907,"[deserve to be hung, wish death would come to ...",[kill myself]
592,P00908,"[blow my brains out, wish death would come to ...",[kill myself]
593,P00909,"[wanna die, wish death would come to me, kill ...",[kill myself]
600,P00925,"[please kill me, deserve nothing but death, wa...",[please kill me]
601,P00926,"[please kill me, deserve nothing but death, wa...",[please kill me]
615,P00955,"[planning to ... commit suicide, took a knife ...",[planning to ... commit suicide]
626,P00966,"[sharp objects ... tempting, wonder how it wou...",[wonder how it would feel ... through my heart]
725,P01096,[I don't know how I'm going to do it],[I don't know how I'm going to do it]


## Configuration

MacBook defaults are conservative: batch size 1 with gradient accumulation. If memory is insufficient, set `max_length=768`. On a Colab NVIDIA GPU, try `train_batch_size=2`, `eval_batch_size=4`, and `max_length=1536`.

For a first experiment, train one grouped fold. After architecture and thresholds are stable, run all five folds with different `validation_fold` values and ensemble them during the later leaderboard stage.

In [4]:
cfg = Config(
    model_name='answerdotai/ModernBERT-base',
    max_length=1024,
    epochs=5,
    train_batch_size=1,
    eval_batch_size=2,
    grad_accumulation=8,
    learning_rate=2e-5,
    head_learning_rate=1e-4,
    validation_fold=0,
    n_splits=5,
    seed=42,
    patience=2,
    output_dir='outputs/transformer_multitask/fold_0_seed_42',
)
fit_idx, val_idx = grouped_split(frame, cfg)
print('Training rows:', len(fit_idx))
print('Validation rows:', len(val_idx))
print('Shared authors:', set(frame.iloc[fit_idx].anon_user_id) & set(frame.iloc[val_idx].anon_user_id))
cfg

Training rows: 1305
Validation rows: 330
Shared authors: set()


Config(model_name='answerdotai/ModernBERT-base', max_length=1024, epochs=5, train_batch_size=1, eval_batch_size=2, grad_accumulation=8, learning_rate=2e-05, head_learning_rate=0.0001, weight_decay=0.01, warmup_ratio=0.1, patience=2, seed=42, validation_fold=0, n_splits=5, risk_loss_weight=0.4, evidence_loss_weight=0.3, factor_loss_weight=0.3, factor_gamma_neg=4.0, factor_gamma_pos=1.0, evidence_positive_weight=8.0, evidence_max_spans=3, output_dir='outputs/transformer_multitask/fold_0_seed_42', num_workers=0)

## Train and evaluate

This downloads the open model on its first run. On an Apple-silicon MacBook, training can take a long time. Do not close the notebook while the cell is running. The best checkpoint and thresholds are saved after each improvement.

In [5]:
result = train_one_fold(TRAIN_PATH, cfg)
result

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/wang/Downloads/bigdata-main/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/wang/Downloads/bigdata-main/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this be

{
  "epoch": 1,
  "train_loss": 0.8658919638715028,
  "risk_weighted_f1": 0.4566076798425105,
  "phrase_f1": 0.5622589531680441,
  "factor_macro_f1": 0.26415343676704744,
  "composite": 0.4305667889175317
}
{
  "epoch": 2,
  "train_loss": 0.5565200714886873,
  "risk_weighted_f1": 0.6590181949891719,
  "phrase_f1": 0.6660396169487078,
  "factor_macro_f1": 0.3183329110096926,
  "composite": 0.5589190363831888
}
{
  "epoch": 3,
  "train_loss": 0.3566861169662512,
  "risk_weighted_f1": 0.7265393715720844,
  "phrase_f1": 0.7137380296471205,
  "factor_macro_f1": 0.3518034064088192,
  "composite": 0.6102781794456156
}
{
  "epoch": 4,
  "train_loss": 0.24954958601472935,
  "risk_weighted_f1": 0.7134225277931231,
  "phrase_f1": 0.7226702085792994,
  "factor_macro_f1": 0.3565374461460009,
  "composite": 0.6091313075348394
}
{
  "epoch": 5,
  "train_loss": 0.1991537203990865,
  "risk_weighted_f1": 0.7399854625661078,
  "phrase_f1": 0.7226702085792994,
  "factor_macro_f1": 0.36415880121145966,
  "

{'best_composite': 0.6220428879636708,
 'history': [{'epoch': 1,
   'train_loss': 0.8658919638715028,
   'risk_weighted_f1': 0.4566076798425105,
   'phrase_f1': 0.5622589531680441,
   'factor_macro_f1': 0.26415343676704744,
   'composite': 0.4305667889175317},
  {'epoch': 2,
   'train_loss': 0.5565200714886873,
   'risk_weighted_f1': 0.6590181949891719,
   'phrase_f1': 0.6660396169487078,
   'factor_macro_f1': 0.3183329110096926,
   'composite': 0.5589190363831888},
  {'epoch': 3,
   'train_loss': 0.3566861169662512,
   'risk_weighted_f1': 0.7265393715720844,
   'phrase_f1': 0.7137380296471205,
   'factor_macro_f1': 0.3518034064088192,
   'composite': 0.6102781794456156},
  {'epoch': 4,
   'train_loss': 0.24954958601472935,
   'risk_weighted_f1': 0.7134225277931231,
   'phrase_f1': 0.7226702085792994,
   'factor_macro_f1': 0.3565374461460009,
   'composite': 0.6091313075348394},
  {'epoch': 5,
   'train_loss': 0.1991537203990865,
   'risk_weighted_f1': 0.7399854625661078,
   'phrase_f1

## Inspect the best validation metrics

In [6]:
metrics_path = Path(cfg.output_dir) / 'best_metrics.json'
best = json.loads(metrics_path.read_text())
print('Risk weighted F1:', best['risk_weighted_f1'])
print('Evidence Phrase F1:', best['phrase_f1'])
print('Factor Macro F1:', best['factor_macro_f1'])
print('Composite:', best['composite'])
print('\nRisk classification report:\n')
print(best['classification_report'])
display(pd.DataFrame({
    'factor': FACTOR_LABELS,
    'threshold': [best['factor_thresholds'][x] for x in FACTOR_LABELS],
}).sort_values('threshold'))

Risk weighted F1: 0.7399854625661078
Evidence Phrase F1: 0.7226702085792994
Factor Macro F1: 0.36415880121145966
Composite: 0.6220428879636708

Risk classification report:

              precision    recall  f1-score   support

   Indicator     0.8029    0.8943    0.8462       123
    Ideation     0.7054    0.7524    0.7281       105
    Behavior     0.6667    0.5641    0.6111        78
     Attempt     0.8667    0.5417    0.6667        24

    accuracy                         0.7455       330
   macro avg     0.7604    0.6881    0.7130       330
weighted avg     0.7443    0.7455    0.7400       330



,factor,threshold
18,sexual orientation related issues,0.050
13,exposure to others' suicide,0.325
6,poor school performance,0.325
2,substance use,0.375
4,emotion dysregulation,0.450
9,prior self-harm or suicidal thought/attempt,0.450
0,mental health issues,0.475
19,social support,0.475
11,interpersonal difficulty,0.475
1,physical health/characteristic,0.475


## Optional full cross-validation — run later

For an honest out-of-fold estimate and a future five-model ensemble, repeat training for folds 0–4. This is intentionally not executed automatically because it is expensive on a MacBook.

```python
all_results = []
for fold in range(5):
    fold_cfg = Config(**{**cfg.__dict__,
        'validation_fold': fold,
        'output_dir': f'outputs/transformer_multitask/fold_{fold}_seed_42',
    })
    all_results.append(train_one_fold(TRAIN_PATH, fold_cfg))
```

The single saved fold can now be used for a diagnostic leaderboard submission without retraining. A five-fold ensemble remains the stronger later option.

## Task 2 optimization — run this next

This trains a **separate label-wise attention factor model** initialized from the encoder you already trained. It does not repeat risk/evidence training and does not alter `best_model.pt`. Every factor learns its own attention over the post.

In [2]:
from pathlib import Path
import importlib
import transformer_multitask as tm
tm = importlib.reload(tm)

factor_result = tm.train_factor_label_attention(
    train_path=Path('train.xlsx'),
    source_checkpoint_dir=Path('outputs/transformer_multitask/fold_0_seed_42'),
    output_dir=Path('outputs/factor_label_attention/fold_0_seed_42'),
    epochs=5,
    patience=2,
)
factor_result

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized encoder; missing=0, unexpected=0
{
  "epoch": 1,
  "train_loss": 0.5871881446961699,
  "factor_macro_f1": 0.39244609306261946
}
{
  "epoch": 2,
  "train_loss": 0.4050217256235437,
  "factor_macro_f1": 0.4456583313664164
}
{
  "epoch": 3,
  "train_loss": 0.3029736424671393,
  "factor_macro_f1": 0.4639538233132658
}
{
  "epoch": 4,
  "train_loss": 0.23054810522473862,
  "factor_macro_f1": 0.47247831610421526
}
{
  "epoch": 5,
  "train_loss": 0.1991632272314523,
  "factor_macro_f1": 0.46803558893033176
}


{'best_factor_macro_f1': 0.47247831610421526,
 'history': [{'epoch': 1,
   'train_loss': 0.5871881446961699,
   'factor_macro_f1': 0.39244609306261946},
  {'epoch': 2,
   'train_loss': 0.4050217256235437,
   'factor_macro_f1': 0.4456583313664164},
  {'epoch': 3,
   'train_loss': 0.3029736424671393,
   'factor_macro_f1': 0.4639538233132658},
  {'epoch': 4,
   'train_loss': 0.23054810522473862,
   'factor_macro_f1': 0.47247831610421526},
  {'epoch': 5,
   'train_loss': 0.1991632272314523,
   'factor_macro_f1': 0.46803558893033176}],
 'output_dir': 'outputs/factor_label_attention/fold_0_seed_42',
 'source_checkpoint_unchanged': 'outputs/transformer_multitask/fold_0_seed_42'}

## Inspect optimized Task 2 result

Run this only after the preceding factor-training cell finishes. We will compare it with the old factor Macro F1 of `0.3642` before building any submission.

In [4]:

import json
factor_metrics_path = Path('outputs/factor_label_attention/fold_0_seed_42/best_factor_metrics.json')
factor_best = json.loads(factor_metrics_path.read_text())
print('Optimized Task 2 Macro F1:', factor_best['factor_macro_f1'])
print('Previous Task 2 Macro F1: 0.3641588012')
print('Improvement:', factor_best['factor_macro_f1'] - 0.3641588012)

Optimized Task 2 Macro F1: 0.47247831610421526
Previous Task 2 Macro F1: 0.3641588012
Improvement: 0.10831951490421526


## Diagnostic leaderboard submission — no retraining

**Do not run this yet.** It currently uses the older factor head. After the optimized Task 2 result is reviewed, this cell will be updated to combine the saved risk/evidence model with the new factor model.

In [5]:
from pathlib import Path
import importlib
import transformer_multitask as tm
tm = importlib.reload(tm)  # expose inference code added after training

CHECKPOINT_DIR = Path('outputs/transformer_multitask/fold_0_seed_42')
LEADERBOARD_PATH = Path('leaderboard.xlsx')
SUBMISSION_PATH = Path('outputs/YBJC.csv')  # change YourTeamName

submission = tm.predict_leaderboard_from_checkpoint(
    CHECKPOINT_DIR,
    LEADERBOARD_PATH,
    SUBMISSION_PATH,
)
print('Saved:', SUBMISSION_PATH.resolve())
print('Rows:', len(submission))
display(submission['risk_level'].value_counts().to_frame('count'))
display(submission.head(10))

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: /Users/wang/Downloads/bigdata-main/outputs/YBJC.csv
Rows: 378


,count
risk_level,
Indicator,136
Ideation,114
Behavior,98
Attempt,30


,row_id,risk_level,evidence,factors
0,P00008,Ideation,want to kill myself; feel like killing myself ...,"[""mental health issues"", ""physical health/char..."
1,P00009,Ideation,want to kill myself; feel like killing myself ...,"[""mental health issues"", ""physical health/char..."
2,P00010,Ideation,want to kill myself; feel like killing myself ...,"[""mental health issues"", ""physical health/char..."
3,P00011,Ideation,want to kill myself; feel like killing myself ...,"[""mental health issues"", ""physical health/char..."
4,P00012,Ideation,want to kill myself; feel like killing myself ...,"[""mental health issues"", ""physical health/char..."
5,P00013,Ideation,wanted to cut him off; cut off an online; feel...,"[""mental health issues"", ""physical health/char..."
6,P00014,Indicator,,"[""mental health issues"", ""substance use"", ""hop..."
7,P00156,Ideation,im killing myself anyways; not wish to live an...,"[""substance use"", ""hopelessness"", ""emotion dys..."
8,P00157,Behavior,jump to my death; if i kill myself; jump,"[""mental health issues"", ""hopelessness"", ""emot..."
9,P00158,Attempt,want; try anymore,"[""emotion dysregulation"", ""suicide means (with..."
